<a href="https://colab.research.google.com/github/stardust-96/Learning-Concepts/blob/main/Semi_Supervised_Contrastive_Learning_CIFAR_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Pre-reqs

In [ ]:
!pip install torch torchvision

In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# 2. Data and Augmentation

In [ ]:
class SimCLRTransform:
    def __init__(self):
        self.base = transforms.Compose([
            transforms.RandomResizedCrop(32, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
            transforms.ToTensor()
        ])

    def __call__(self, x):
        return self.base(x), self.base(x)


In [ ]:
dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=SimCLRTransform()
)

loader = DataLoader(dataset, batch_size=256, shuffle=True)

100%|██████████| 170M/170M [00:15<00:00, 11.2MB/s]


# SimCLR

In [ ]:
class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = models.resnet18(weights=None)
        self.encoder.fc = nn.Identity()   # remove classifier

        self.projector = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 128)
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return F.normalize(z, dim=1)


# CL (NT-Xent)

In [ ]:
def contrastive_loss(z1, z2, temperature=0.5):
    z = torch.cat([z1, z2], dim=0)
    sim = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=2)

    batch_size = z1.size(0)
    labels = torch.arange(batch_size)
    labels = torch.cat([labels, labels], dim=0).to(z.device)

    sim = sim / temperature
    loss = F.cross_entropy(sim, labels)
    return loss


# Training Loop

In [ ]:
model = SimCLR().cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

for epoch in range(5):
    for (x1, x2), _ in loader:
        x1, x2 = x1.cuda(), x2.cuda()

        z1 = model(x1)
        z2 = model(x2)

        loss = contrastive_loss(z1, z2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch}, loss = {loss.item():.4f}")


Epoch 0, loss = 3.5812
Epoch 1, loss = 3.5714
Epoch 2, loss = 3.5271
Epoch 3, loss = 3.4980
Epoch 4, loss = 3.5090
